# VIF for dataset with abrahams parameters (S9/Table S10)

In [7]:
from pathlib import Path

import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

from training.train_utils import get_data, ModelSettings, DataSettings

X, y, _ = get_data(DataSettings(0.85, True, 'all_water', True, 'name'), ModelSettings('rf', 'abrahams', 'kf'), Path("data"))

X = X.astype({col: int for col in X.columns if col.startswith('sorb')})#.loc[:,[col for col in X.columns if not col.startswith('abraha')]]
X_wo_abr = X.loc[:,[col for col in X.columns if not col.startswith('abraha')]]
abr = X.loc[:,[col for col in X.columns if col.startswith('abraha')]]
X_wo_abr_ph = X_wo_abr.drop(columns="ph")
vif = {}
for (name, dat) in (
    ('full data', X), ('data w/o abrahams', X_wo_abr), ('data w/o abrahams and ph', X_wo_abr_ph), ('only abrahams', abr),
):
    for i, col_name in enumerate(dat.columns):
        vif[col_name] = vif.get(col_name, [])
        vif[col_name].append(variance_inflation_factor(StandardScaler().fit_transform(dat), i))
    for col_name in X.columns.difference(dat.columns):
        vif[col_name] = vif.get(col_name, [])
        vif[col_name].append("-")
df = pd.DataFrame(vif).T
df.columns = ['all data', 'data w/o Abrahams', 'data w/o Abrahams and pH', 'Abrahams']
print(df)

                          all data data w/o Abrahams data w/o Abrahams and pH  \
ph                       13.525414          8.397879                        -   
SLR (kg/L)                4.270683          2.251918                 2.204075   
SSA (m^2/g)               1.873612           1.77471                 1.675652   
particle size             1.401962          1.346981                  1.34453   
T                         1.664904          1.494599                 1.464637   
low conc_range (µg/L)     3.972735          2.884464                 2.705909   
high conc_range (µg/L)    6.073852          4.157011                 4.151483   
seawater                 13.164355          9.058996                 2.926854   
sorb_PC                   1.179663          1.138351                 1.138323   
sorb_PE                   8.708304          8.304221                 8.057148   
sorb_PLA                  1.959329          1.645236                 1.645059   
sorb_PMMA                 1.

In [8]:
tbl_lines = [
    r"\begin{center}",
    r"  \begin{longtblr}[",
    r"    caption={Variance inflation factors for full and selected subsets of the data}",
    r"  ]{colspec={l||X[c]|X[c]|X[c]|X[c]}, rowhead=1, column{1-5}={fg=blue}}",
    r"    parameter & all data & all data w/o Abrahams & all data w/o Abrahams and pH & Abrahams \\",
    r"    \hline",
]

def _highlight(val):
    if isinstance(val, str):
        return val
    val = val.round(2)
    if val > 10:
        return r"\textbf{\textit{" + f"{val}" +r"}}"
    elif val > 3:
        return r"\textit{" + f"{val}" +r"}"
    return val

for row in df.itertuples():
    idx = row[0]
    tbl_lines.append(f"    {idx} & {_highlight(row[1])} & {_highlight(row[2])} & {_highlight(row[3])} & {_highlight(row[4])}" + r" \\")
    if idx == "seawater" or idx == "sorb_PVC":
        tbl_lines.append(r"    \hline[dashed]")
tbl_lines.append(r"  \end{longtblr}")
tbl_lines.append(r"\end{center}")
print("\n".join(map(lambda x: x.replace("_", r"\_"), tbl_lines)))
# tbl_lines.append(r"\end{table}")

\begin{center}
  \begin{longtblr}[
    caption={Variance inflation factors for full and selected subsets of the data}
  ]{colspec={l||X[c]|X[c]|X[c]|X[c]}, rowhead=1, column{1-5}={fg=blue}}
    parameter & all data & all data w/o Abrahams & all data w/o Abrahams and pH & Abrahams \\
    \hline
    ph & \textbf{\textit{13.53}} & \textit{8.4} & - & - \\
    SLR (kg/L) & \textit{4.27} & 2.25 & 2.2 & - \\
    SSA (m^2/g) & 1.87 & 1.77 & 1.68 & - \\
    particle size & 1.4 & 1.35 & 1.34 & - \\
    T & 1.66 & 1.49 & 1.46 & - \\
    low conc\_range (µg/L) & \textit{3.97} & 2.88 & 2.71 & - \\
    high conc\_range (µg/L) & \textit{6.07} & \textit{4.16} & \textit{4.15} & - \\
    seawater & \textbf{\textit{13.16}} & \textit{9.06} & 2.93 & - \\
    \hline[dashed]
    sorb\_PC & 1.18 & 1.14 & 1.14 & - \\
    sorb\_PE & \textit{8.71} & \textit{8.3} & \textit{8.06} & - \\
    sorb\_PLA & 1.96 & 1.65 & 1.65 & - \\
    sorb\_PMMA & 1.18 & 1.13 & 1.13 & - \\
    sorb\_PP & 2.99 & 2.83 & 2.81 & - \\
   

# Filter Details (S14/Table S12)

In [5]:
from pathlib import Path

from training.isotherm_filters import has_enough_datapoints, has_enough_ooms, is_linear_fit, is_non_linear_fit, has_enough_ooms_and_dp
from training.train_utils import get_data, DataSettings, ModelSettings
tbl_lines = [
    r"\begin{center}",
    r"  \begin{tblr}[",
    r"    caption={Details on target range and salinity and isotherm fit distributions in selected subsets of the data}",
    r"  ]{colspec={l||X[c]|X[c]|X[c]|X[c]}, rowhead=1, column{1-5}={fg=blue}}",
    r"    name & salinities & isotherm fit & target range & description \\",
    r"    \hline",
]

for name, filt in [
    ('all data', lambda _: True), ('freshwater', lambda row: not row['seawater']),
    ('seawater', lambda row: row['seawater']), ('ooms', has_enough_ooms),
    ('linear fit', is_linear_fit), ('non-linear fit', is_non_linear_fit)
]:
    X, y, _ = get_data(DataSettings(0.85, False, 'all_water', True, 'name'), ModelSettings(None, 'raw', 'kf'), Path("data"), filt=['iso_fit_as_bool', filt])
    _, y_n, _ = get_data(DataSettings(0.85, False, 'all_water', True, 'name'), ModelSettings(None, 'raw', 'n'), Path("data"), filt=[filt])
    kf = y.max() - y.min()
    n = y_n.max() - y_n.min()
    seawater = "" if name == 'freshwater' else f"seawater: {X[X['seawater']].shape[0]}"
    freshwater = "" if name == 'seawater' else f"freshwater: {X[~X['seawater']].shape[0]}"
    lin = "" if filt.__name__ == 'is_non_linear_fit' else f"linear: {X[X['iso_fit_linear']].shape[0]}"
    nlin = "" if filt.__name__ == 'is_linear_fit' else f"non-linear: {X[~X['iso_fit_linear']].shape[0]}"
    tbl_lines.append(f"    " + r"\SetCell[r=2]{} " + name + f"& {seawater} & {lin} & "
                     + r"$\log(K_f)$: " + f"{kf:.2f} & " + r"\SetCell[r=2]{} " + r"\\")
    tbl_lines.append(f"    " + f"& {freshwater} & {nlin} & n: {n:.2f} & " + r"\\")
    if name == 'seawater':
        tbl_lines.append(r"    \hline[dashed]")

tbl_lines.append(r"  \end{tblr}")
tbl_lines.append(r"\end{center}")
print("\n".join(tbl_lines))

\begin{center}
  \begin{tblr}[
    caption={Details on target range and salinity and isotherm fit distributions in selected subsets of the data}
  ]{colspec={l||X[c]|X[c]|X[c]|X[c]}, rowhead=1, column{1-5}={fg=blue}}
    name & salinities & isotherm fit & target range & description \\
    \hline
    \SetCell[r=2]{} all data& seawater: 103 & linear: 130 & $\log(K_f)$: 11.35 & \SetCell[r=2]{} \\
    & freshwater: 131 & non-linear: 104 & n: 1.77 & \\
    \SetCell[r=2]{} freshwater&  & linear: 39 & $\log(K_f)$: 8.91 & \SetCell[r=2]{} \\
    & freshwater: 131 & non-linear: 92 & n: 1.60 & \\
    \SetCell[r=2]{} seawater& seawater: 103 & linear: 91 & $\log(K_f)$: 8.42 & \SetCell[r=2]{} \\
    &  & non-linear: 12 & n: 1.75 & \\
    \hline[dashed]
    \SetCell[r=2]{} ooms& seawater: 103 & linear: 130 & $\log(K_f)$: 11.35 & \SetCell[r=2]{} \\
    & freshwater: 121 & non-linear: 94 & n: 1.75 & \\
    \SetCell[r=2]{} linear fit& seawater: 91 & linear: 130 & $\log(K_f)$: 11.35 & \SetCell[r=2]{} \\
